# Análise de Dados de Delivery

Este notebook contém o pipeline analítico completo estruturado em 8 etapas:
1. **Análise Exploratória de Dados (EDA)** dos pedidos
2. **Criação de Novas Colunas (Feature Engineering)** - Cálculo da Receita do Item
3. **Tratamento de Valores Ausentes** - Preenchimento da quantidade e remoção de registros com preço nulo
4. **Agregações por Item** - Quantidade total, receita total e identificação dos Top 5 itens
5. **Análise Temporal** - Conversão de data, extração de mês e evolução de vendas
6. **Integração de Dados (Merge)** - Junção com cardápio e análise por Categoria
7. **Filtros e Consultas** - Pedidos de 'Salgados' com mais de 10 unidades
8. **KPIs e Análise Estatística com NumPy** - Receita total, ticket médio e percentis

## 0. Importações e Configurações

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Configurações de exibição do Pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

## 1. Análise Exploratória de Dados (EDA)

Carregando o arquivo `pedidos.csv` e explorando sua estrutura através de `head()`, `tail()`, `info()` e `describe()`.

In [ ]:
# Carregando o arquivo de pedidos
caminho_pedidos = Path("dados") / "pedidos.csv"
df_pedidos = pd.read_csv(caminho_pedidos)

# Dimensões do dataset (linhas x colunas)
linhas, colunas = df_pedidos.shape
print(f"Quantidade de registros (linhas): {linhas}")
print(f"Quantidade de colunas: {colunas}")

### Primeiros Registros (`head`)

In [ ]:
df_pedidos.head()

### Últimos Registros (`tail`)

In [ ]:
df_pedidos.tail()

### Informações Estruturais e Tipos de Dados (`info`)

In [ ]:
df_pedidos.info()

### Estatísticas Descritivas (`describe`)

In [ ]:
df_pedidos.describe(include="all")

### Resumo de Tipos de Dados e Valores Ausentes (Nulos)

In [ ]:
resumo_colunas = pd.DataFrame({
    "Tipo": df_pedidos.dtypes,
    "Não-Nulos": df_pedidos.notnull().sum(),
    "Nulos": df_pedidos.isnull().sum(),
    "% Nulos": (df_pedidos.isnull().sum() / len(df_pedidos) * 100).round(2)
})
resumo_colunas

## 2. Criação de Novas Colunas (Feature Engineering)

Nesta etapa, calculamos a **receita de cada item do pedido** multiplicando a quantidade pelo preço unitário:

$$\text{Receita\_Item} = \text{Quantidade} \times \text{Preco\_Unitario}$$

E adicionamos a nova coluna `Receita_Item` ao DataFrame.

In [ ]:
# Criando a coluna Receita_Item inicial
df_pedidos["Receita_Item"] = df_pedidos["Quantidade"] * df_pedidos["Preco_Unitario"]

# Exibindo os primeiros registros com a nova coluna criada
df_pedidos[["ID_Pedido", "Item", "Quantidade", "Preco_Unitario", "Receita_Item"]].head(10)

### Resumo Estatístico da Nova Coluna (`Receita_Item`)

In [ ]:
df_pedidos[["Receita_Item"]].describe()

## 3. Tratamento de Valores Ausentes

Nesta etapa, tratamos os valores ausentes identificados:
1. **Identificação dos nulos** detalhadamente por coluna
2. **Preenchimento** dos valores ausentes da coluna `Quantidade` com a sua média
3. **Remoção** das linhas que contenham valores nulos na coluna `Preco_Unitario`
4. **Recálculo** da coluna `Receita_Item` para eliminar inconsistências

### 3.1 Identificação de Valores Nulos nas Colunas

In [ ]:
print("Quantidade de valores nulos por coluna:")
total_nulos = df_pedidos.isnull().sum()
perc_nulos = (df_pedidos.isnull().sum() / len(df_pedidos) * 100).round(2)
tabela_nulos = pd.DataFrame({"Total Nulos": total_nulos, "% Nulos": perc_nulos})
tabela_nulos

### 3.2 Preenchimento de Valores Ausentes em `Quantidade` com a Média

In [ ]:
# Cálculo da média da coluna Quantidade
media_quantidade = df_pedidos["Quantidade"].mean()
print(f"Média calculada para Quantidade: {media_quantidade:.2f}")

# Preenchimento dos valores nulos com a média
df_pedidos["Quantidade"] = df_pedidos["Quantidade"].fillna(media_quantidade)

# Verificando que não restaram nulos em Quantidade
print(f"Valores nulos restantes em Quantidade: {df_pedidos['Quantidade'].isnull().sum()}")

### 3.3 Remoção de Linhas com Valores Nulos em `Preco_Unitario`

In [ ]:
linhas_antes = len(df_pedidos)

# Removendo linhas onde Preco_Unitario é nulo
df_pedidos = df_pedidos.dropna(subset=["Preco_Unitario"]).copy()

linhas_depois = len(df_pedidos)
print(f"Linhas antes da remoção: {linhas_antes}")
print(f"Linhas removidas: {linhas_antes - linhas_depois}")
print(f"Linhas restantes: {linhas_depois}")

### 3.4 Recálculo da `Receita_Item` e Verificação Final de Nulos

In [ ]:
# Recalculando a coluna Receita_Item com os dados limpos
df_pedidos["Receita_Item"] = df_pedidos["Quantidade"] * df_pedidos["Preco_Unitario"]

# Verificação final de valores ausentes em todo o DataFrame
print("Valores nulos restantes no DataFrame após o tratamento:")
print(df_pedidos.isnull().sum())

# Visualização dos primeiros registros do dataset limpo
df_pedidos.head(10)

## 4. Agregações por Item

Nesta etapa, agrupamos os dados por cada item vendido para calcular:
- **Quantidade total vendida** de cada item
- **Receita total** gerada por cada item
- **Top 5 itens mais vendidos** (por volume/quantidade)
- **Top 5 itens que geraram mais receita** (faturamento)

### 4.1 Agrupamento Geral por Item

In [ ]:
# Agrupamento por Item com agregações de Quantidade e Receita
df_itens = df_pedidos.groupby("Item").agg(
    Quantidade_Total=("Quantidade", "sum"),
    Receita_Total=("Receita_Item", "sum"),
    Total_Pedidos=("ID_Pedido", "count")
).reset_index()

# Arredondando métricas financeiras e volumétricas
df_itens["Quantidade_Total"] = df_itens["Quantidade_Total"].round(2)
df_itens["Receita_Total"] = df_itens["Receita_Total"].round(2)

# Exibindo todos os 20 itens agregados
df_itens.sort_values(by="Item")

### 4.2 Top 5 Itens Mais Vendidos (por Quantidade)

In [ ]:
# Ordenando de forma decrescente pela quantidade total
top5_quantidade = df_itens.sort_values(by="Quantidade_Total", ascending=False).head(5)

print("=== TOP 5 ITENS MAIS VENDIDOS (POR QUANTIDADE) ===")
top5_quantidade

### 4.3 Top 5 Itens que Geraram Mais Receita

In [ ]:
# Ordenando de forma decrescente pela receita total
top5_receita = df_itens.sort_values(by="Receita_Total", ascending=False).head(5)

print("=== TOP 5 ITENS COM MAIOR RECEITA TOTAL ===")
top5_receita

## 5. Análise Temporal

Nesta etapa, analisamos a sazonalidade e o faturamento histórico:
- Conversão da coluna `Data` para o tipo `datetime`
- Extração do mês e período (`Ano_Mes`)
- Cálculo da receita total mensal
- Análise da evolução das vendas ao longo do tempo

### 5.1 Conversão da Data e Extração de Mês e Ano-Mês

In [ ]:
# Convertendo a coluna Data para o tipo datetime
df_pedidos["Data"] = pd.to_datetime(df_pedidos["Data"])

# Extraindo o mês (1-12) e o período Ano-Mês para agregação temporal contínua
df_pedidos["Mes"] = df_pedidos["Data"].dt.month
df_pedidos["Ano_Mes"] = df_pedidos["Data"].dt.to_period("M")

# Exibindo tipos e exemplos de datas
print("Tipo da coluna Data após conversão:", df_pedidos["Data"].dtype)
df_pedidos[["ID_Pedido", "Data", "Mes", "Ano_Mes", "Receita_Item"]].head()

### 5.2 Receita Total Mensal e Evolução ao Longo do Tempo

In [ ]:
# Agrupando por período cronológico (Ano-Mês)
receita_mensal = df_pedidos.groupby("Ano_Mes").agg(
    Receita_Total=("Receita_Item", "sum"),
    Total_Itens=("Quantidade", "sum"),
    Total_Pedidos=("ID_Pedido", "count")
).reset_index()

receita_mensal["Receita_Total"] = receita_mensal["Receita_Total"].round(2)
receita_mensal["Total_Itens"] = receita_mensal["Total_Itens"].round(2)
receita_mensal["Ticket_Medio_Mensal"] = (receita_mensal["Receita_Total"] / receita_mensal["Total_Pedidos"]).round(2)

print("Evolução Mensal da Receita e Vendas:")
receita_mensal

### 5.3 Destaques da Evolução Temporal

In [ ]:
mes_maior_receita = receita_mensal.loc[receita_mensal["Receita_Total"].idxmax()]
mes_menor_receita = receita_mensal.loc[receita_mensal["Receita_Total"].idxmin()]
media_receita_mensal = receita_mensal["Receita_Total"].mean()

print(f"Mês com MAIOR receita: {mes_maior_receita['Ano_Mes']} com R$ {mes_maior_receita['Receita_Total']:,.2f}")
print(f"Mês com MENOR receita: {mes_menor_receita['Ano_Mes']} com R$ {mes_menor_receita['Receita_Total']:,.2f}")
print(f"Receita média mensal: R$ {media_receita_mensal:,.2f}")

## 6. Integração de Dados (Merge)

Nesta etapa, enriquecemos a base de pedidos cruzando-a com as informações do cardápio:
- Carregamento de `cardapio.csv`
- Merge (junção) com `pedidos` através da coluna `Item`
- Agrupamento e cálculo da receita total por `Categoria`
- Identificação da categoria líder em vendas

In [ ]:
# Carregando o arquivo cardapio.csv
caminho_cardapio = Path("dados") / "cardapio.csv"
df_cardapio = pd.read_csv(caminho_cardapio)

# Visualizando o cardápio de referência
print("Cardápio de Referência:")
display(df_cardapio.head())

# Merge dos pedidos com o cardápio usando a coluna 'Item'
df_completo = pd.merge(df_pedidos, df_cardapio, on="Item", how="left")

print(f"Dimensões após o merge: {df_completo.shape}")
df_completo.head(10)

### 6.1 Receita Total por Categoria e Identificação da Categoria Líder

In [ ]:
# Agrupamento por Categoria
receita_categoria = df_completo.groupby("Categoria").agg(
    Receita_Total=("Receita_Item", "sum"),
    Quantidade_Total=("Quantidade", "sum"),
    Total_Pedidos=("ID_Pedido", "count")
).reset_index()

# Calculando a participação percentual de cada categoria no faturamento
receita_categoria["Receita_Total"] = receita_categoria["Receita_Total"].round(2)
receita_categoria["Quantidade_Total"] = receita_categoria["Quantidade_Total"].round(2)
receita_categoria["% Faturamento"] = (receita_categoria["Receita_Total"] / receita_categoria["Receita_Total"].sum() * 100).round(2)

# Ordenando por maior faturamento
receita_categoria = receita_categoria.sort_values(by="Receita_Total", ascending=False)

print("=== RECEITA TOTAL POR CATEGORIA ===")
display(receita_categoria)

cat_lider = receita_categoria.iloc[0]
print(f"\n🏆 Categoria que mais gera receita: {cat_lider['Categoria']} (R$ {cat_lider['Receita_Total']:,.2f} - {cat_lider['% Faturamento']}%")

## 7. Filtros e Consultas

Filtrando os pedidos que atendem aos seguintes critérios de negócio:
- Pertencem à categoria **'Salgados'**
- Possuem **quantidade superior a 10 unidades**

In [ ]:
# Aplicação dos critérios de filtro
filtro_salgados_acima_10 = df_completo[
    (df_completo["Categoria"] == "Salgados") & (df_completo["Quantidade"] > 10)
]

print(f"Total de pedidos encontrados: {len(filtro_salgados_acima_10)}")
print(f"Percentual da base total: {(len(filtro_salgados_acima_10) / len(df_completo) * 100):.2f}%")

# Exibindo os primeiros resultados filtrados
filtro_salgados_acima_10[["ID_Pedido", "Data", "Item", "Categoria", "Quantidade", "Preco_Unitario", "Receita_Item"]].head(15)

## 8. KPIs e Análise Estatística com NumPy

Cálculo dos principais indicadores-chave de desempenho (KPIs) da operação:
- **Receita Total**: soma de toda a receita gerada
- **Total de Itens Vendidos**: soma de todas as quantidades vendidas
- **Ticket Médio**: receita total dividida pelo total de pedidos
- **Desafio Extra**: Cálculo dos percentis (25%, 50% e 75%) de `Preco_Unitario` e `Quantidade` utilizando a biblioteca **NumPy**

In [ ]:
# Cálculo dos KPIs operacionais
receita_total = df_completo["Receita_Item"].sum()
total_itens_vendidos = df_completo["Quantidade"].sum()
total_pedidos = df_completo["ID_Pedido"].nunique()
ticket_medio = receita_total / total_pedidos

# Exibição formatada dos KPIs
kpis = pd.DataFrame({
    "KPI": ["Receita Total", "Total de Itens Vendidos", "Total de Pedidos", "Ticket Médio"],
    "Valor": [
        f"R$ {receita_total:,.2f}",
        f"{total_itens_vendidos:,.2f} unid",
        f"{total_pedidos}",
        f"R$ {ticket_medio:,.2f}"
    ]
})

print("=== INDICADORES PRINCIPAIS DO NEGÓCIO (KPIS) ===")
display(kpis)

### Desafio Extra: Análise de Percentis com NumPy (`np.percentile`)

In [ ]:
# Percentis de Preco_Unitario usando NumPy
percentis_preco = np.percentile(df_completo["Preco_Unitario"], [25, 50, 75])

# Percentis de Quantidade usando NumPy
percentis_quantidade = np.percentile(df_completo["Quantidade"], [25, 50, 75])

# Tabela consolidada de percentis
tabela_percentis = pd.DataFrame({
    "Percentil": ["25% (Q1)", "50% (Mediana / Q2)", "75% (Q3)"],
    "Preco_Unitario": [f"R$ {percentis_preco[0]:.2f}", f"R$ {percentis_preco[1]:.2f}", f"R$ {percentis_preco[2]:.2f}"],
    "Quantidade": [f"{percentis_quantidade[0]:.2f} unid", f"{percentis_quantidade[1]:.2f} unid", f"{percentis_quantidade[2]:.2f} unid"]
})

print("=== PERCENTIS CALCULADOS COM NUMPY ===")
display(tabela_percentis)

## Resumo Consolidado das Etapas (1 a 8)

### Q&A
- **Qual a evolução temporal da receita?** As vendas abrangem de janeiro de 2023 a setembro de 2025. O pico histórico ocorreu em **Fevereiro de 2023** (R$ 6.646,03), com faturamento médio mensal em torno de **R$ 3.716,75**.
- **Qual categoria gera maior receita?** A categoria **Salgados** é a líder absoluta, com **R$ 53.073,30** (43,27% da receita total do delivery).
- **Quantos pedidos de 'Salgados' tiveram mais de 10 unidades?** Foram encontrados **109 pedidos**, representando mais de 25% do total de transações.
- **Quais os principais KPIs do negócio?**
  - Receita Total: **R$ 122.652,59**
  - Total de Itens Vendidos: **6.833,25 unidades**
  - Ticket Médio por Pedido: **R$ 285,24**
- **Quais são os percentis calculados com NumPy?**
  - Preço Unitário: 25% = R$ 8,92 | 50% = R$ 13,10 | 75% = R$ 27,40
  - Quantidade: 25% = 8,00 | 50% = 16,00 | 75% = 24,00

### Data Analysis Key Findings
- A categoria **Salgados** concentra a maior fatia do faturamento (43,27%), impulsionada pelas pizzas (Calabresa e Mussarela) e hambúrgueres.
- As categorias **Doces** (R$ 22.293,94) e **Oriental** (R$ 21.599,80) ocupam o segundo e terceiro lugares, demonstrando relevância nas compras complementares e pratos especializados.
- A distribuição de quantidade indica que metade dos pedidos (mediana = 16 unidades) envolve volumes médios a altos, característicos de pedidos corporativos ou em família.

### Insights or Next Steps
- Fortalecer cross-selling entre Salgados e Bebidas (atualmente Bebidas responde por apenas 7,46% da receita).
- Criar promoções específicas nos meses historicamente de menor receita (como Março e Abril) para estabilizar o fluxo de faturamento ao longo do ano.